<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 29 · RNN, LSTM, GRU</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Tự cài RNN, đo gradient biến mất và gradient nổ qua thời gian, kẹp gradient, so RNN với LSTM và GRU trên bài nhớ ký hiệu đầu, rồi huấn luyện một mô hình ngôn ngữ ký tự trên Truyện Kiều.</p>
</div>

### Sau notebook này bạn sẽ

- Tự viết vòng lặp RNN và khớp nn.RNN.
- Đo gradient qua thời gian và dùng kẹp gradient.
- Thấy bias cổng quyết định LSTM, GRU có nhớ được xa hay không.
- Huấn luyện mô hình ngôn ngữ ký tự và sinh thơ lục bát theo nhiệt độ.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch29-rnn-lstm-gru/01-du-lieu-chuoi-va-rnn/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Dữ liệu chuỗi và RNN](#trang-1)
2. [Trang 2 · Gradient qua thời gian](#trang-2)
3. [Trang 3 · LSTM và GRU](#trang-3)
4. [Trang 4 · Mô hình ngôn ngữ ký tự trên Truyện Kiều](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Dữ liệu chuỗi và RNN

nn.RNN và vòng lặp tự cài.

*Đọc kèm trên web: [Trang 1 · Dữ liệu chuỗi và RNN](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch29-rnn-lstm-gru/01-du-lieu-chuoi-va-rnn/)*

In [ ]:
import torch                                                # PyTorch
import torch.nn as nn                                       # các lớp mạng
torch.manual_seed(0)                                        # khởi tạo lặp lại được
rnn = nn.RNN(input_size=3, hidden_size=4, batch_first=True) # shape (lô, bước, đặc trưng)
x = torch.randn(2, 5, 3)                                    # 2 chuỗi, 5 bước, 3 đặc trưng
ra, h_cuoi = rnn(x)                                         # trạng thái mọi bước, trạng thái cuối
print(ra.shape, h_cuoi.shape)                               # (2, 5, 4) và (1, 2, 4)
print([(n, tuple(p.shape)) for n, p in rnn.named_parameters()])   # W_x, W_h, hai bias

Output mong đợi:

```text
torch.Size([2, 5, 4]) torch.Size([1, 2, 4])
[('weight_ih_l0', (4, 3)), ('weight_hh_l0', (4, 4)), ('bias_ih_l0', (4,)), ('bias_hh_l0', (4,))]
```

In [ ]:
h = torch.zeros(2, 4)                                       # h_0 = 0
cac_h = []                                                  # trạng thái từng bước
for t in range(5):                                          # đọc lần lượt từng bước
    h = torch.tanh(x[:, t] @ rnn.weight_ih_l0.T + rnn.bias_ih_l0 + h @ rnn.weight_hh_l0.T + rnn.bias_hh_l0)   # công thức RNN
    cac_h.append(h)                                         # lưu lại
tu_cai = torch.stack(cac_h, dim=1)                          # (2, 5, 4)
print(torch.allclose(tu_cai, ra, atol=1e-6), torch.allclose(h, h_cuoi[0], atol=1e-6))   # so với nn.RNN

Output mong đợi:

```text
True True
```

In [ ]:
print(sum(p.numel() for p in rnn.parameters()), rnn(torch.randn(2, 500, 3))[0].shape)   # 36 tham số, chuỗi 500 bước

Output mong đợi:

```text
36 torch.Size([2, 500, 4])
```

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Gradient qua thời gian

Gradient theo khoảng cách, gradient nổ, kẹp gradient.

*Đọc kèm trên web: [Trang 2 · Gradient qua thời gian](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch29-rnn-lstm-gru/02-gradient-qua-thoi-gian/)*

In [ ]:
def do_grad(mang, T, seed=0):                               # gradient của h_T theo từng x_t
    torch.manual_seed(seed)                                 # cùng chuỗi vào
    x = torch.randn(1, T, 8, requires_grad=True)            # một chuỗi T bước
    ra, _ = mang(x)                                         # lượt xuôi
    ra[0, -1].sum().backward()                              # loss giả: tổng trạng thái cuối
    return x.grad[0].norm(dim=1)                            # một số cho mỗi bước
torch.manual_seed(0)                                        # seed
r = nn.RNN(8, 32, batch_first=True)                         # khởi tạo mặc định
g = do_grad(r, 100)                                         # chuỗi 100 bước
print([f"{g[t].item():.1e}" for t in (99, 90, 50, 0)])      # gần cuối tới bước đầu
print([f"{do_grad(r, T)[0].item():.1e}" for T in (2, 10, 20, 50, 100)])   # về bước đầu theo độ dài

Output mong đợi:

```text
['2.0e+00', '5.5e-03', '3.0e-14', '0.0e+00']
['9.6e-01', '3.4e-03', '1.3e-05', '1.0e-13', '0.0e+00']
```

In [ ]:
with torch.no_grad():                                       # sửa trọng số trực tiếp
    r.weight_hh_l0 *= 3                                     # phóng to ma trận hồi quy
print([f"{do_grad(r, T)[0].item():.1e}" for T in (2, 10, 20, 50, 100)])   # gradient lớn dần

Output mong đợi:

```text
['2.3e+00', '1.5e+00', '1.4e+01', '1.8e+01', '6.1e+01']
```

In [ ]:
w = nn.Parameter(torch.zeros(3))                            # một tham số giả
w.grad = torch.tensor([30.0, 40.0, 0.0])                    # gradient độ dài 50
print(nn.utils.clip_grad_norm_([w], max_norm=1.0).item(), [round(v, 4) for v in w.grad.tolist()])   # độ dài trước khi kẹp, gradient sau khi kẹp

Output mong đợi:

```text
50.0 [0.6, 0.8, 0.0]
```

<a name="trang-3" id="trang-3"></a>

## Trang 3 · LSTM và GRU

LSTM, GRU và bài nhớ ký hiệu đầu (khoảng 2 phút trên CPU).

*Đọc kèm trên web: [Trang 3 · LSTM và GRU](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch29-rnn-lstm-gru/03-lstm-va-gru/)*

In [ ]:
for loai in (nn.RNN, nn.LSTM, nn.GRU):                      # ba loại lớp hồi quy
    print(loai.__name__, sum(p.numel() for p in loai(8, 32).parameters()))   # số tham số

Output mong đợi:

```text
RNN 1344
LSTM 5376
GRU 4032
```

In [ ]:
H = 32                                                      # kích thước ẩn
for b in (0.0, 1.0, 3.0):                                   # cộng thêm vào bias cổng quên
    torch.manual_seed(0)                                    # cùng khởi tạo
    lstm = nn.LSTM(8, H, batch_first=True)                  # LSTM mặc định
    with torch.no_grad():                                   # sửa trọng số trực tiếp
        lstm.bias_ih_l0[H:2 * H] += b                       # khối thứ hai là cổng quên (i, f, g, o)
    f = torch.sigmoid(lstm.bias_ih_l0[H:2 * H] + lstm.bias_hh_l0[H:2 * H]).mean().item()   # cổng quên trung bình lúc đầu
    print(b, round(f, 3), [f"{do_grad(lstm, T)[0].item():.1e}" for T in (2, 10, 20, 50, 100)])   # gradient về bước đầu

Output mong đợi:

```text
0.0 0.507 ['2.3e-01', '1.7e-03', '2.2e-05', '2.7e-11', '7.1e-22']
1.0 0.736 ['3.4e-01', '2.4e-02', '5.8e-03', '5.3e-05', '1.7e-09']
3.0 0.953 ['4.3e-01', '1.9e-01', '2.6e-01', '6.7e-02', '4.4e-02']
```

In [ ]:
def tao_lo(n, T, g):                                        # n chuỗi T ký hiệu 0..9, nhãn là ký hiệu đầu
    X = torch.randint(0, 10, (n, T), generator=g)           # ký hiệu ngẫu nhiên
    return X, X[:, 0].clone()                               # chuỗi, nhãn
X, y = tao_lo(2, 12, torch.Generator().manual_seed(0))      # hai chuỗi ví dụ
print(X.tolist(), y.tolist())                               # nhãn là cột đầu

Output mong đợi:

```text
[[4, 9, 3, 0, 3, 9, 7, 3, 7, 3, 1, 6], [6, 9, 8, 6, 6, 8, 4, 3, 6, 9, 1, 4]] [4, 6]
```

In [ ]:
class Nho(nn.Module):                                       # embedding, lớp hồi quy, Linear
    def __init__(self, loai, bias_cong=0.0):                # loai: nn.RNN, nn.LSTM hoặc nn.GRU
        super().__init__()                                  # bắt buộc
        self.nhung = nn.Embedding(10, 16)                   # ký hiệu thành vector 16 số
        self.loi = loai(16, 64, batch_first=True)           # lớp hồi quy 64 chiều
        self.ra = nn.Linear(64, 10)                         # 10 lớp
        if bias_cong:                                       # LSTM: cổng quên; GRU: cổng cập nhật z
            with torch.no_grad():                           # sửa trọng số trực tiếp
                self.loi.bias_ih_l0[64:128] += bias_cong    # khối thứ hai của bias
    def forward(self, x):                                   # lượt xuôi
        ra, _ = self.loi(self.nhung(x))                     # trạng thái mọi bước
        return self.ra(ra[:, -1])                           # chỉ dùng bước cuối
ce = nn.CrossEntropyLoss()                                  # loss
def huan_luyen(loai, bias_cong=0.0, kep=None, T=50, so_buoc=1500):   # huấn luyện và in độ chính xác
    torch.manual_seed(0)                                    # cùng khởi tạo
    m = Nho(loai, bias_cong)                                # mô hình
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)         # Adam
    g = torch.Generator().manual_seed(1)                    # nguồn lô train
    Xte, yte = tao_lo(2000, T, torch.Generator().manual_seed(2))   # 2000 chuỗi test cố định
    acc, g_max = [], 0.0                                    # kết quả
    for b in range(1, so_buoc + 1):                         # mỗi bước
        x, y = tao_lo(64, T, g)                             # một lô mới
        opt.zero_grad()                                     # xóa gradient
        ce(m(x), y).backward()                              # lượt ngược
        do_dai = nn.utils.clip_grad_norm_(m.parameters(), kep or float("inf"))   # kep=None: chỉ đo
        g_max = max(g_max, do_dai.item())                   # gradient lớn nhất từng gặp
        opt.step()                                          # cập nhật
        if b % 250 == 0:                                    # mỗi 250 bước
            with torch.no_grad():                           # đánh giá
                acc.append(round((m(Xte).argmax(1) == yte).float().mean().item(), 3))   # độ chính xác test
    print(loai.__name__, bias_cong, kep, acc, round(g_max, 1))   # một dòng kết quả
for loai in (nn.RNN, nn.LSTM, nn.GRU):                      # ba loại, khởi tạo mặc định
    huan_luyen(loai)                                        # chuỗi 50 bước

Output mong đợi:

```text
RNN 0.0 None [0.105, 0.094, 0.091, 0.1, 0.101, 0.095] 0.6
LSTM 0.0 None [0.101, 0.087, 0.097, 0.101, 0.101, 0.102] 0.2
GRU 0.0 None [0.101, 0.089, 0.099, 0.1, 0.096, 0.102] 0.3
```

In [ ]:
huan_luyen(nn.LSTM, bias_cong=3.0)                          # cổng quên mở sẵn
huan_luyen(nn.GRU, bias_cong=3.0)                           # cổng cập nhật mở sẵn

Output mong đợi:

```text
LSTM 3.0 None [0.746, 0.867, 0.988, 1.0, 0.899, 1.0] 140.0
GRU 3.0 None [0.231, 0.999, 1.0, 1.0, 1.0, 1.0] 53.2
```

In [ ]:
huan_luyen(nn.LSTM, bias_cong=3.0, kep=1.0)                 # thêm kẹp gradient ở 1,0

Output mong đợi:

```text
LSTM 3.0 1.0 [0.697, 0.983, 0.999, 1.0, 0.994, 0.994] 723.6
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Thử bias_cong=1.0 cho LSTM. Có đủ cho chuỗi 50 bước không?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Đổi lr của Adam thành 3e-3 với LSTM bias 3. Cú rơi độ chính xác có nhiều hơn không?</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Mô hình ngôn ngữ ký tự trên Truyện Kiều

Mô hình ngôn ngữ ký tự trên Truyện Kiều (khoảng 2 phút trên CPU, nhanh hơn nếu bật GPU).

*Đọc kèm trên web: [Trang 4 · Mô hình ngôn ngữ ký tự trên Truyện Kiều](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch29-rnn-lstm-gru/04-mo-hinh-ngon-ngu-ky-tu/)*

In [ ]:
import re                                                   # biểu thức chính quy
import urllib.request                                       # tải trang web
URL = "https://vi.wikisource.org/w/index.php?oldid=205132&action=raw"   # Truyện Kiều trên Wikisource, ghim một phiên bản
yc = urllib.request.Request(URL, headers={"User-Agent": "ai-and-robotics-handbook/1.0 (giao trinh hoc AI)"})   # Wikimedia cần User-Agent
tho = urllib.request.urlopen(yc).read().decode("utf-8")     # mã nguồn trang
than = tho.split("<poem>")[1].split("</poem>")[0]           # phần thơ
cau = [re.sub(r"\{\{số\|\d+\}\}", "", d).strip() for d in than.splitlines()]   # bỏ số thứ tự câu
cau = [c for c in cau if c]                                 # bỏ dòng trống
print(len(cau), cau[0], "/", cau[1])                        # số câu, hai câu đầu
print(sum(len(c.split()) == (6 if k % 2 == 0 else 8) for k, c in enumerate(cau)))   # số câu đúng nhịp 6-8

Output mong đợi:

```text
3254 Trăm năm trong cõi người ta, / Chữ tài chữ mệnh khéo là ghét nhau.
3254
```

In [ ]:
import math                                                 # log, exp
van = "\n".join(cau) + "\n"                               # cả bài thơ, xuống dòng cũng là ký tự
chu = sorted(set(van))                                      # bảng ký tự
so = {c: i for i, c in enumerate(chu)}                      # ký tự thành số
du_lieu = torch.tensor([so[c] for c in van])                # cả văn bản thành số
cat = int(0.9 * len(du_lieu))                               # chia theo vị trí
tr, va = du_lieu[:cat], du_lieu[cat:]                       # 90% đầu học, 10% cuối đánh giá
print(len(van), len(chu), len(tr), len(va))                 # kích thước
dem = torch.bincount(tr, minlength=len(chu)).float()        # tần suất từng ký tự trong train
print(round(math.log(len(chu)), 3), round(-torch.log(dem[va] / dem.sum()).mean().item(), 3))   # hai mốc loss

Output mong đợi:

```text
104204 132 93783 10421
4.883 3.59
```

In [ ]:
import copy                                                 # chép trạng thái mô hình
class MoHinhKyTu(nn.Module):                                # embedding, LSTM, Linear
    def __init__(self, V, H=256):                           # V ký tự, ẩn H
        super().__init__()                                  # bắt buộc
        self.nhung = nn.Embedding(V, 64)                    # ký tự thành vector 64 số
        self.lstm = nn.LSTM(64, H, batch_first=True)        # LSTM 256 chiều
        self.ra = nn.Linear(H, V)                           # logit cho mọi ký tự
    def forward(self, x, tt=None):                          # tt: trạng thái (h, c) mang theo khi sinh
        h, tt = self.lstm(self.nhung(x), tt)                # trạng thái mọi bước
        return self.ra(h), tt                               # đoán ký tự tiếp theo ở mọi bước
def loss_val(m):                                            # loss trên toàn bộ tập val
    m.eval()                                                # chế độ đánh giá
    with torch.no_grad():                                   # không cần gradient
        lg, _ = m(va[:-1].unsqueeze(0))                     # cả tập val như một chuỗi
    m.train()                                               # trở lại huấn luyện
    return ce(lg[0], va[1:]).item()                         # cross-entropy trung bình
torch.manual_seed(0)                                        # khởi tạo lặp lại được
m = MoHinhKyTu(len(chu))                                    # mô hình
opt = torch.optim.Adam(m.parameters(), lr=3e-3)             # Adam
g = torch.Generator().manual_seed(1)                        # nguồn vị trí ngẫu nhiên
L, B = 128, 64                                              # độ dài đoạn, số đoạn mỗi lô
tot_nhat, ban_tot = float("inf"), None                      # bản tốt nhất
for b in range(1, 1001):                                    # 1000 bước
    i = torch.randint(0, len(tr) - L - 1, (B,), generator=g)   # 64 vị trí bắt đầu
    x = torch.stack([tr[j:j + L] for j in i])               # đầu vào
    y = torch.stack([tr[j + 1:j + L + 1] for j in i])       # đích: lệch phải 1 ký tự
    lg, _ = m(x)                                            # logit (64, 128, 132)
    opt.zero_grad()                                         # xóa gradient
    ce(lg.reshape(-1, len(chu)), y.reshape(-1)).backward()  # loss trên mọi bài đoán
    nn.utils.clip_grad_norm_(m.parameters(), 1.0)           # kẹp gradient
    opt.step()                                              # cập nhật
    if b % 100 == 0:                                        # mỗi 100 bước
        l = loss_val(m)                                     # đo trên val
        print(b, round(l, 3))                               # in ra
        if l < tot_nhat:                                    # tốt hơn bản đã giữ
            tot_nhat, ban_tot = l, copy.deepcopy(m.state_dict())   # giữ lại
m.load_state_dict(ban_tot)                                  # dùng bản tốt nhất
print(sum(p.numel() for p in m.parameters()), round(tot_nhat, 3), round(math.exp(tot_nhat), 2))   # tham số, loss, perplexity

Output mong đợi:

```text
100 1.887
200 1.703
300 1.622
400 1.586
500 1.567
600 1.576
700 1.586
800 1.627
900 1.677
1000 1.725
372100 1.567 4.79
```

In [ ]:
def sinh(m, dau, n, nhiet, seed=0):                         # sinh n ký tự sau đoạn mở đầu
    torch.manual_seed(seed)                                 # lấy mẫu lặp lại được
    m.eval()                                                # chế độ đánh giá
    ra = dau                                                # văn bản sinh ra
    with torch.no_grad():                                   # không cần gradient
        lg, tt = m(torch.tensor([[so[c] for c in dau]]))    # đọc đoạn mở đầu
        for _ in range(n):                                  # mỗi ký tự mới
            p = torch.softmax(lg[0, -1] / nhiet, dim=0)     # nhiệt độ chia logit
            k = torch.multinomial(p, 1)                     # lấy mẫu một ký tự
            ra += chu[k.item()]                             # nối vào
            lg, tt = m(k.view(1, 1), tt)                    # chỉ đưa ký tự mới, mang trạng thái
    return ra                                               # kết quả
print(sinh(m, "Trăm năm trong cõi người ta,\n", 300, 0.8)) # 300 ký tự ở nhiệt độ 0,8

Output mong đợi:

```text
Trăm năm trong cõi người ta,
Có lâu mây thấy thảy vào,
Xuân thu khuyết thấy xuân tàn,
Máu suy lại gió con đọng thâm.
Nàng rằng: Hiếc dắc vạnh thì thôi!
Li liều vâng tấm thì bời,
Dưới rưung đủ đã rõ rầm nghe thuy,
Trong thuyền ví lắm chi nàng thân.
Dặm nghe viện trả, trong này,
Sởn nghe nàng cũng này trời mai.
Bây giờ sách nhớ c
```

In [ ]:
for nhiet in (0.2, 1.2):                                    # hai đầu thang nhiệt độ
    print(nhiet, sinh(m, "Trăm năm trong cõi người ta,\n", 150, nhiet).split("\n")[1:4])   # ba dòng đầu sinh ra

Output mong đợi:

```text
0.2 ['Cho nghe nghe những nghĩa cho người cho thang.', 'Chàng chi chẳng thấy một ngày một người.', 'Nghĩ lời nghĩ chẳng thiệt đào,']
1.2 ['Ôi lây mẹ Giám tình.', 'Bay giờ mực, cóm nhi thân,', 'Trước nào phục tủi việt tinh,']
```

In [ ]:
tieng_kieu = {w.strip(",.;:!?\"'").lower() for c in cau for w in c.split()}   # mọi tiếng có trong Truyện Kiều
for nhiet in (0.2, 0.5, 0.8, 1.2):                          # bốn nhiệt độ
    dong = sinh(m, "Trăm năm trong cõi người ta,\n", 3000, nhiet).split("\n")[1:-1]   # bỏ câu mở đầu, dòng dở cuối
    so_tieng = [len(d.split()) for d in dong]               # số tiếng mỗi dòng
    tieng = [w.strip(",.;:!?\"'").lower() for d in dong for w in d.split()]   # mọi tiếng sinh ra
    print(nhiet, len(dong), round(sum(s in (6, 8) for s in so_tieng) / len(dong), 3),   # tỷ lệ dòng 6 hoặc 8 tiếng
          round(sum(w in tieng_kieu for w in tieng) / len(tieng), 3), sum(d in cau for d in dong))   # tiếng có thật, dòng chép nguyên văn

Output mong đợi:

```text
0.2 80 0.637 0.998 0
0.5 85 0.741 0.984 0
0.8 91 0.571 0.929 0
1.2 95 0.663 0.816 0
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Đổi đoạn mở đầu thành một câu 6 tiếng của bạn. Câu tiếp theo có 8 tiếng không?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Thay nn.LSTM bằng nn.GRU trong MoHinhKyTu. Loss val tốt nhất thay đổi thế nào?</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Đoán trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch29-rnn-lstm-gru/05-bai-tap/)*

### Bài 29.1 · LSTM hai lớp

Shape của ra, h_n, c_n và số tham số của nn.LSTM(16, 64, num_layers=2).
<details><summary>Xem đáp án</summary>

ra (8, 20, 64), h_n và c_n (2, 8, 64), 54 272 tham số. Code ở ô ngay dưới.
</details>

In [ ]:
lstm2 = nn.LSTM(16, 64, num_layers=2, batch_first=True)     # hai lớp chồng lên nhau
ra, (h_n, c_n) = lstm2(torch.randn(8, 20, 16))              # LSTM trả về một cặp (h, c)
print(ra.shape, h_n.shape, c_n.shape)                       # shape
print(sum(p.numel() for p in lstm2.parameters()), [(n, p.numel()) for n, p in lstm2.named_parameters() if "weight" in n])   # tham số

Output mong đợi:

```text
torch.Size([8, 20, 64]) torch.Size([2, 8, 64]) torch.Size([2, 8, 64])
54272 [('weight_ih_l0', 4096), ('weight_hh_l0', 16384), ('weight_ih_l1', 16384), ('weight_hh_l1', 16384)]
```

### Bài 29.2 · Nhớ qua 100 bước

LSTM bias cổng quên 3, kẹp 1,0, chuỗi 100 bước.
<details><summary>Xem đáp án</summary>

99,9% sau 1500 bước nhưng chậm hơn chuỗi 50 bước nhiều. Code ở ô ngay dưới.
</details>

In [ ]:
huan_luyen(nn.LSTM, bias_cong=3.0, kep=1.0, T=100)          # chuỗi dài gấp đôi

Output mong đợi:

```text
LSTM 3.0 1.0 [0.327, 0.518, 0.668, 0.814, 0.975, 0.999] 1168.3
```

### Bài 29.3 · Nhiệt độ bằng tay

Xác suất của logit [2, 1, 0] ở nhiệt độ 1; 0,5; 2.
<details><summary>Xem đáp án</summary>

0,665 / 0,245 / 0,09 ở nhiệt độ 1. Code ở ô ngay dưới.
</details>

In [ ]:
z = torch.tensor([2.0, 1.0, 0.0])                           # ba logit
for nhiet in (1.0, 0.5, 2.0):                               # ba nhiệt độ
    print(nhiet, [round(v, 3) for v in torch.softmax(z / nhiet, 0).tolist()])   # xác suất

Output mong đợi:

```text
1.0 [0.665, 0.245, 0.09]
0.5 [0.867, 0.117, 0.016]
2.0 [0.506, 0.307, 0.186]
```